Ячейка 1: Инициализация проекта и импорт библиотек

In [19]:
import os
import json
import re
import pandas as pd
from pathlib import Path
from datetime import datetime

# Определяем пути к данным
RAW_DATA_DIR = Path("../data/raw_things/")
OUTPUT_DIR = Path("../data/output/")

# Создаем папку для выгрузки, если её еще нет
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Форматируем текущую дату и время (ГодМесяцДень_ЧасыМинутыСекунды)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_file = OUTPUT_DIR / f"guns_result_{timestamp}.csv"

print(f"Библиотеки импортированы. Рабочие директории настроены.")
print(f"Файл при экспорте будет сохранен как: {output_file.name}")

Библиотеки импортированы. Рабочие директории настроены.
Файл при экспорте будет сохранен как: guns_result_20260719_011753.csv


Ячейка 2: Загрузка основных баз данных

In [20]:
# Загружаем богатый файл баланса предметов
inventory_path = RAW_DATA_DIR / "NexusConfigStoreInventory.json"
with open(inventory_path, "r", encoding="utf-8") as f:
    inventory_data = json.load(f)

# Загружаем переводчик имен
name_parts_path = RAW_DATA_DIR / "NexusConfigStoreInventoryNamePart.json"
with open(name_parts_path, "r", encoding="utf-8") as f:
    name_parts_data = json.load(f)

# Автоматически строим карту перевода для ВСЕХ физических запчастей игры
part_translation_map = {}
for category, cat_val in inventory_data.items():
    if not isinstance(cat_val, dict):
        continue
    parts_dict = cat_val.get("parts", {})
    if not isinstance(parts_dict, dict):
        continue
        
    for part_id, part_val in parts_dict.items():
        if not isinstance(part_val, dict):
            continue
        part_name = part_val.get("name", "")
        # Нас интересуют только физические детали
        if not isinstance(part_name, str) or not part_name.startswith("part_"):
            continue
        
        fields = part_val.get("fields", {})
        if not isinstance(fields, dict):
            continue
        
        # Сканируем аспекты детали в поисках InventoryNamingAspect
        aspects = fields.get("Aspects", [])
        for aspect in aspects:
            if not isinstance(aspect, dict):
                continue
            if "InventoryNamingAspect" in aspect.get("structtype", ""):
                title_list = aspect.get("TitlePartList", []) or []
                prefix_list = aspect.get("PrefixPartList", []) or []
                suffix_list = aspect.get("SuffixPartList", []) or []
                
                all_naming_defs = title_list + prefix_list + suffix_list
                for naming_def in all_naming_defs:
                    if isinstance(naming_def, str):
                        # Вытаскиваем ключ (например, "InventoryNamePartDef'np_Zipgun'" -> "np_Zipgun")
                        match = re.search(r"'(np_.*?)'", naming_def)
                        if match:
                            np_key = match.group(1)
                            # Находим этот ключ в базе локализации
                            if np_key in name_parts_data:
                                real_part_name = name_parts_data[np_key].get("fields", {}).get("PartName", "")
                                if real_part_name:
                                    # Сохраняем перевод для полного и сокращенного технического имени детали
                                    part_translation_map[part_name.lower()] = real_part_name
                                    part_translation_map[part_name.replace("part_", "").lower()] = real_part_name

print(f"Базы данных загружены! Построен автоматический словарь перевода деталей: {len(part_translation_map)} записей.")

Базы данных загружены! Построен автоматический словарь перевода деталей: 778 записей.


Ячейка 3: Глубокий сбор легендарных предметов и их свойств

In [21]:
legendary_items = []

# Разрешенные типы оружия (исключили тяжелое оружие HW)
weapon_types = ["PS", "SR", "AR", "SG", "SM"]

# Проходимся по всем категориям в богатом файле Inventory.json
for category, cat_val in inventory_data.items():
    # Защита: cat_val должен быть словарем
    if not isinstance(cat_val, dict):
        continue
        
    if "Weapon" in category and category == "1 | Weapon":
        continue 
        
    parts_dict = cat_val.get("parts", {})
    
    # Защита: parts_dict должен быть словарем, а не списком []
    if not isinstance(parts_dict, dict):
        continue
        
    for part_id, part_val in parts_dict.items():
        # Защита: каждый отдельный компонент part_val тоже должен быть словарем
        if not isinstance(part_val, dict):
            continue
            
        part_path = part_val.get("path", "")
        # Проверяем, что путь — это действительно строка
        if not isinstance(part_path, str):
            continue
            
        # Фильтруем легендарные (comp_05_legendary) или перламутровые (comp_06_pearl) компоненты
        is_legendary = "comp_05_legendary" in part_path
        is_pearlescent = "comp_06_pearl" in part_path
        
        if is_legendary or is_pearlescent:
            fields = part_val.get("fields", {})
            if not isinstance(fields, dict):
                continue
                
            is_exclude = fields.get("bExcludeFromGlobalPool", False)
            world_drop_flag = not is_exclude
            
            selection_rules = fields.get("PartTypeSelectionRules", {})
            if not isinstance(selection_rules, dict):
                selection_rules = {}
            
            # Определяем редкость в зависимости от найденного маркера
            rarity_str = "Pearlescent" if is_pearlescent else "Legendary"
            
            # Определяем тип по категории (например, DAD_PS -> PS)
            item_type_suffix = category.split("_")[-1] if "_" in category else "Unknown"
            
            # ФИЛЬТР: только обычные пушки
            if item_type_suffix.upper() not in weapon_types:
                continue
            
            legendary_items.append({
                "Item_Code": part_path,
                "Internal_Category": category,
                "Type": item_type_suffix,
                "Rarity": rarity_str,
                "World_Drop": world_drop_flag,
                "Manufacturer": "Unknown",
                "Display_Name": "Unknown",
                "Drop_Source": "Unknown",
                "Selection_Rules": selection_rules # Сохраняем правила для разбора по слотам
            })

df = pd.DataFrame(legendary_items)
df = df.drop_duplicates(subset=["Item_Code"]).reset_index(drop=True)

print(f"Инициализация завершена. Безопасно собрано легендарных и перламутровых ПУШЕК: {len(df)}")
df.head()

Инициализация завершена. Безопасно собрано легендарных и перламутровых ПУШЕК: 160


,Item_Code,Internal_Category,Type,Rarity,World_Drop,Manufacturer,Display_Name,Drop_Source,Selection_Rules
0,DAD_PS.comp_05_legendary_Zipgun,2 | DAD_PS,PS,Legendary,True,Unknown,Unknown,Unknown,"{'barrel': {'PartCount': {'min': 1, 'MAX': 1},..."
1,DAD_PS.comp_05_legendary,2 | DAD_PS,PS,Legendary,False,Unknown,Unknown,Unknown,{}
2,DAD_PS.comp_05_legendary_rangefinder,2 | DAD_PS,PS,Legendary,True,Unknown,Unknown,Unknown,"{'barrel': {'PartCount': {'min': 1, 'MAX': 1},..."
3,DAD_PS.comp_05_legendary_soulsurvivor,2 | DAD_PS,PS,Legendary,False,Unknown,Unknown,Unknown,"{'barrel': {'PartCount': {'min': 1, 'MAX': 1},..."
4,JAK_PS.comp_05_legendary,3 | JAK_PS,PS,Legendary,False,Unknown,Unknown,Unknown,{}


Ячейка 4: Определение производителей и типов

In [22]:
# Словарь для перевода аббревиатур типов оружия в красивые английские названия
type_mapping = {
    "PS": "Pistol", "SR": "Sniper Rifle", "AR": "Assault Rifle", 
    "SG": "Shotgun", "SM": "Submachine Gun", "HW": "Heavy Weapon",
    "SHIELD": "Shield", "GRENADE": "Grenade", "CLASSMOD": "Class Mod", "ARTIFACT": "Artifact"
}

# Словарь соответствия префиксов и полных названий производителей
mfr_mapping = {
    "DAD": "Daedalus", "ORD": "Order", "BORG": "Ripper", "BOR": "Ripper",
    "JAK": "Jakobs", "VLA": "Vladof", "MAL": "Maliwan", "HYP": "Hyperion",
    "TED": "Tediore", "TOR": "Torgue", "COV": "CoV", "ATL": "Atlas"
}

# Сама функция определения производителя по коду предмета
def determine_manufacturer(item_code):
    if not isinstance(item_code, str):
        return "Unknown"
    # Извлекаем префикс перед первым нижним подчеркиванием (например, DAD_PS... -> DAD)
    prefix = item_code.split("_")[0].upper()
    # Очищаем от возможных системных кавычек
    prefix = prefix.replace("INV'", "").replace("'", "")
    return mfr_mapping.get(prefix, "Unknown")

# Применяем сопоставление производителей
df["Manufacturer"] = df["Item_Code"].apply(determine_manufacturer)

# Переводим сокращения типов в красивые английские названия
df["Type"] = df["Type"].str.upper().map(type_mapping).fillna(df["Type"])

print("Производители и типы успешно обновлены!")
# Посмотрим, как теперь выглядят эти колонки
df[["Item_Code", "Type", "Manufacturer"]].head()

Производители и типы успешно обновлены!


,Item_Code,Type,Manufacturer
0,DAD_PS.comp_05_legendary_Zipgun,Pistol,Daedalus
1,DAD_PS.comp_05_legendary,Pistol,Daedalus
2,DAD_PS.comp_05_legendary_rangefinder,Pistol,Daedalus
3,DAD_PS.comp_05_legendary_soulsurvivor,Pistol,Daedalus
4,JAK_PS.comp_05_legendary,Pistol,Jakobs


Ячейка 5: Строгая расшифровка названий

In [23]:
# Сама функция расшифровки имен (теперь она всегда будет в памяти этой ячейки)
def resolve_display_name(item_code, name_parts):
    if not isinstance(item_code, str):
        return "Unknown"
        
    # Ищем суффикс после легендарного или перламутрового маркера
    match = re.search(r"comp_05_legendary_(.*)", item_code, re.IGNORECASE)
    if not match:
        match = re.search(r"comp_06_pearl_(.*)", item_code, re.IGNORECASE)
    if not match:
        match = re.search(r"legendary_(.*)", item_code, re.IGNORECASE)
    if not match:
        match = re.search(r"pearl_(.*)", item_code, re.IGNORECASE)
        
    if match:
        raw_name = match.group(1).lower().replace("_", "") # Приводим к единому виду (например, "zipgun")
        
        # Ищем строгое соответствие в ключах name_parts (где ключи типа "np_absolution")
        for np_key, np_val in name_parts.items():
            # Очищаем ключ от префикса "np_" и нижних подчеркиваний
            clean_np_key = np_key.lower().replace("np_", "").replace("_", "")
            
            # Если очищенные ключи полностью совпадают
            if clean_np_key == raw_name:
                # Забираем PartName, если его нет — пишем Unknown
                return np_val.get("fields", {}).get("PartName", "Unknown")
                
    return "Unknown"

# Применяем строгую функцию расшифровки имен к нашему DataFrame
df["Display_Name"] = df.apply(lambda row: resolve_display_name(row["Item_Code"], name_parts_data), axis=1)

print("Названия предметов успешно расшифрованы!")
# Посмотрим на результат
df[["Item_Code", "Type", "Manufacturer", "Display_Name"]].head(10)

Названия предметов успешно расшифрованы!


,Item_Code,Type,Manufacturer,Display_Name
0,DAD_PS.comp_05_legendary_Zipgun,Pistol,Daedalus,Zipper
1,DAD_PS.comp_05_legendary,Pistol,Daedalus,Unknown
2,DAD_PS.comp_05_legendary_rangefinder,Pistol,Daedalus,Rangefinder
3,DAD_PS.comp_05_legendary_soulsurvivor,Pistol,Daedalus,Soul Survivor
4,JAK_PS.comp_05_legendary,Pistol,Jakobs,Unknown
5,JAK_PS.comp_05_legendary_kingsgambit,Pistol,Jakobs,King's Gambit
6,JAK_PS.comp_05_legendary_phantom_flame,Pistol,Jakobs,Phantom Flame
7,JAK_PS.comp_05_legendary_QuickDraw,Pistol,Jakobs,San Saba Songbird
8,JAK_PS.comp_05_legendary_seventh_sense,Pistol,Jakobs,Seventh Sense
9,JAK_PS.comp_05_legendary_shalashaska,Pistol,Jakobs,Shalashaska


Ячейка 6: Умный расчет шансов босс-дропа и разделение источников

In [24]:
# Загружаем базу пулов добычи
item_pool_list_path = RAW_DATA_DIR / "NexusConfigStoreItemPoolList.json"
with open(item_pool_list_path, "r", encoding="utf-8") as f:
    item_pool_list_data = json.load(f)

drop_sources = {}

# Вспомогательная функция очистки Handle
def clean_handle(handle):
    if not handle or not isinstance(handle, str):
        return ""
    return handle.lower().replace("inv'", "").replace("'", "").strip()

# Сканируем всю базу ItemPoolList
for list_key, list_val in item_pool_list_data.items():
    boss_name = list_key.replace("ItemPoolList_", "").replace("_", " ").title()
    item_pools = list_val.get("fields", {}).get("ItemPools", [])
    
    for pool_entry in item_pools:
        itempool = pool_entry.get("itempool", {})
        item_data = itempool.get("item", {})
        
        # Вариант 1: Вложенный инстанс пула (bInstance == True)
        if item_data.get("bInstance") and "Instance" in item_data:
            instance = item_data["Instance"] or {}
            items_in_pool = instance.get("items", [])
            
            for pool_item in items_in_pool:
                inner_item = pool_item.get("item", {}).get("item", {})
                handle = inner_item.get("Handle")
                
                if handle:
                    cleaned_h = clean_handle(handle)
                    if cleaned_h:
                        if cleaned_h not in drop_sources:
                            drop_sources[cleaned_h] = []
                        drop_sources[cleaned_h].append(boss_name)
                        
        # Вариант 2: Прямая ссылка (bInstance == False)
        else:
            handle = item_data.get("Handle")
            if handle:
                cleaned_h = clean_handle(handle)
                if cleaned_h:
                    if cleaned_h not in drop_sources:
                        drop_sources[cleaned_h] = []
                    drop_sources[cleaned_h].append(boss_name)

# Функция сопоставления для Drop_Source (если босс не найден, выводим прочерк "-")
def get_drop_source(row):
    cleaned_code = clean_handle(row["Item_Code"])
    if not cleaned_code:
        return "-"
    sources = drop_sources.get(cleaned_code, [])
    return ", ".join(set(sources)) if sources else "-"

df["Drop_Source"] = df.apply(get_drop_source, axis=1)

print("Источники дропа успешно привязаны!")

Источники дропа успешно привязаны!


Ячейка 7: Парсинг деталей по индивидуальным слотам пушки и стихиям

In [25]:
# Карта сокращений брендов запчастей к красивым полным названиям
part_brand_map = {
    "jak": "Jakobs",
    "ted": "Tediore",
    "hyp": "Hyperion",
    "cov": "CoV",
    "borg": "Ripper",
    "bor": "Ripper",
    "tor": "Torgue",
    "mal": "Maliwan",
    "vla": "Vladof",
    "atl": "Atlas"
}

def get_part_info(part_code, weapon_manufacturer, translation_map):
    code_lower = part_code.lower()
    cleaned_code = part_code.replace("part_", "")
    
    part_mfr = None
    for suffix, brand_name in part_brand_map.items():
        if f"_{suffix}" in code_lower or f"_{suffix}_" in code_lower:
            part_mfr = brand_name
            break
            
    if not part_mfr:
        part_mfr = weapon_manufacturer
        
    model_name = translation_map.get(part_code.lower(), translation_map.get(cleaned_code.lower(), None))
    
    if model_name:
        return f"{part_mfr} ({model_name})"
    else:
        display_code = cleaned_code
        for suffix in part_brand_map.keys():
            display_code = re.sub(rf"_{suffix}\\b", "", display_code, flags=re.IGNORECASE)
            display_code = re.sub(rf"\\\\b{suffix}_", "", display_code, flags=re.IGNORECASE)
        
        display_code = display_code.replace("_", " ").title()
        return f"{part_mfr} ({display_code})"

def extract_parts_by_slot(selection_rules, slot_keys, weapon_manufacturer, translation_map):
    parts_list = []
    for slot_name, slot_val in selection_rules.items():
        if any(key == slot_name for key in slot_keys):
            parts_in_slot = slot_val.get("parts", [])
            for p in parts_in_slot:
                part_code = p.get("part", "")
                if part_code:
                    formatted_part = get_part_info(part_code, weapon_manufacturer, translation_map)
                    parts_list.append(formatted_part)
                    
    return ", ".join(sorted(list(set(parts_list)))) if parts_list else "-"

# Применяем парсинг слотов с полным 100% покрытием всех категорий из файлов игры
df["Barrels"] = df.apply(lambda row: extract_parts_by_slot(row["Selection_Rules"], ["barrel"], row["Manufacturer"], part_translation_map), axis=1)
df["Grips"] = df.apply(lambda row: extract_parts_by_slot(row["Selection_Rules"], ["grip"], row["Manufacturer"], part_translation_map), axis=1)

# Магазины (объединили все типы магазинов игры)
df["Magazines"] = df.apply(lambda row: extract_parts_by_slot(row["Selection_Rules"], ["magazine", "mag", "body_mag", "magazine_borg"], row["Manufacturer"], part_translation_map), axis=1)

# Прицелы (объединили прицелы и линзы)
df["Scopes"] = df.apply(lambda row: extract_parts_by_slot(row["Selection_Rules"], ["scope", "scope_acc"], row["Manufacturer"], part_translation_map), axis=1)

# Подствольники (объединили подствольные модули и вторичные патроны)
df["Underbarrels"] = df.apply(lambda row: extract_parts_by_slot(row["Selection_Rules"], ["underbarrel", "underbarrel_acc", "secondary_ammo"], row["Manufacturer"], part_translation_map), axis=1)

# Аксессуары (объединили все обвесы, бренд-специфичные детали, а также прошивки и эндгейм-слоты)
df["Accessories"] = df.apply(lambda row: extract_parts_by_slot(row["Selection_Rules"], [
    "body_acc", "barrel_acc", "foregrip", "magazine_acc", 
    "hyperion_secondary_acc", "tediore_acc", "tediore_secondary_acc", 
    "firmware", "endgame"
], row["Manufacturer"], part_translation_map), axis=1)

# Строгий разбор стихий на основе явных параметров в файлах
def extract_elements_advanced(row, inventory_data):
    selection_rules = row.get("Selection_Rules", {})
    category = row.get("Internal_Category", "")
    weapon_manufacturer = row.get("Manufacturer", "")
    
    elements = []
    
    # --- ШАГ 1: Сканируем аспекты выбранных физических деталей пушки ---
    category_parts = inventory_data.get(category, {}).get("parts", {})
    chosen_parts = []
    if isinstance(selection_rules, dict):
        for slot, slot_val in selection_rules.items():
            for p in slot_val.get("parts", []):
                part_name = p.get("part", "")
                if part_name:
                    chosen_parts.append(part_name)
                    
    # Проверяем аспекты каждой выбранной детали на наличие DamageTypeAspect
    for part_code in chosen_parts:
        for p_key, p_val in category_parts.items():
            if isinstance(p_val, dict) and p_val.get("name", "") == part_code:
                aspects = p_val.get("fields", {}).get("Aspects", [])
                for aspect in aspects:
                    if isinstance(aspect, dict):
                        # 1. Проверяем классический DamageTypeAspect
                        if "DamageTypeAspect" in aspect.get("structtype", ""):
                            dmg_type = aspect.get("DamageType", "")
                            if isinstance(dmg_type, str):
                                match = re.search(r"'(.*?)'", dmg_type)
                                if match:
                                    elem_name = match.group(1).title()
                                    if elem_name in ["Fire", "Shock", "Corrosive", "Cryo", "Radiation"]:
                                        elements.append(elem_name)
                                    elif elem_name == "Normal":
                                        elements.append("Kinetic")
                        # 2. Проверяем также WeaponUseModeAspect
                        elif "WeaponUseModeAspect" in aspect.get("structtype", ""):
                            behavior = aspect.get("behavior", {})
                            if isinstance(behavior, dict):
                                dmg_type = behavior.get("DamageType", "")
                                if isinstance(dmg_type, str):
                                    match = re.search(r"'(.*?)'", dmg_type)
                                    if match:
                                        elem_name = match.group(1).title()
                                        if elem_name in ["Fire", "Shock", "Corrosive", "Cryo", "Radiation"]:
                                            elements.append(elem_name)
                                        elif elem_name == "Normal":
                                            elements.append("Kinetic")
    
    # --- ШАГ 2: Собираем правила генерации стихийных слотов из общего базового класса (коммона) ---
    all_rules = {}
    if isinstance(category_parts, dict):
        for part_key, part_val in category_parts.items():
            if isinstance(part_val, dict):
                part_name = part_val.get("name", "")
                if any(x == part_name for x in ["base_comp_01_common", "comp_01_common"]):
                    rules = part_val.get("fields", {}).get("PartTypeSelectionRules", {})
                    if isinstance(rules, dict):
                        all_rules.update(rules)
                        
    if isinstance(selection_rules, dict):
        all_rules.update(selection_rules)
        
    # Ищем любые стихийные слоты (body_ele, secondary_ele, element и т.д.)
    element_slots = {k: v for k, v in all_rules.items() if "ele" in k or "element" in k}
    
    for slot_name, slot_val in element_slots.items():
        part_count = slot_val.get("PartCount", {})
        min_count = part_count.get("min", 0)
        if min_count == 0:
            can_be_kinetic = True
            
        parts_in_slot = slot_val.get("parts", [])
        for p in parts_in_slot:
            part_code = p.get("part", "").lower()
            
            if "fire" in part_code: elements.append("Fire")
            elif "shock" in part_code: elements.append("Shock")
            elif "corrosive" in part_code: elements.append("Corrosive")
            elif "cryo" in part_code: elements.append("Cryo")
            elif "radiation" in part_code: elements.append("Radiation")
            elif "none" in part_code or "phys" in part_code:
                elements.append("Kinetic")
                
    elements = sorted(list(set(elements)))
    
    # Исключаем Kinetic для Maliwan
    if weapon_manufacturer == "Maliwan" and "Kinetic" in elements:
        elements.remove("Kinetic")
        
    # СТРОГОЕ ПРАВИЛО: выводим только то, что нашли в файлах. Если ничего не нашли — пишем прочерк "-"
    if elements:
        return ", ".join(elements)
        
    return "-"

# Применяем исправленный унаследованный разбор стихий
df["Elements"] = df.apply(lambda row: extract_elements_advanced(row, inventory_data), axis=1)

print("Все запчасти, наследуемые стихии и кинетика успешно распределены по колонкам!")

Все запчасти, наследуемые стихии и кинетика успешно распределены по колонкам!


Ячейка 8: Экспорт итоговой таблицы в CSV

In [26]:
final_df = df.copy()

# Переименовываем колонки
final_df = final_df.rename(columns={
    "Display_Name": "Name",
    "Drop_Source": "Drop Source",
    "World_Drop": "World Drop",
    "Item_Code": "Item Code"
})

# УМНАЯ ФИЛЬТРАЦИЯ: Удаляем только пустые базовые шаблоны-заглушки.
# Если код предмета оканчивается строго на "comp_05_legendary" или "comp_06_pearl" — мы его удаляем.
# Если у предмета есть суффикс уникального имени (например, comp_05_legendary_screenwriter), мы его оставляем,
# даже если его имя расшифровалось как "Unknown", чтобы вы могли заметить эту пушку.
is_template = final_df["Item Code"].str.lower().str.endswith(("comp_05_legendary", "comp_06_pearl", "comp_06_pearlescent"))
final_df = final_df[~is_template]

# Задаем финальный порядок колонок: Item Code на первом месте
columns_order = [
    "Item Code", "Name", "Rarity", "Type", "Manufacturer", 
    "World Drop", "Drop Source", "Elements",
    "Barrels", "Grips", "Magazines", "Scopes", "Underbarrels", "Accessories"
]
final_df = final_df[columns_order]

# Экспортируем в CSV-файл (его имя содержит точное время генерации из Ячейки 1)
final_df.to_csv(output_file, index=False, encoding="utf-8")

print(f"Экспорт завершен! Готовая таблица лежит в: {output_file}")
print(f"Размерность: {final_df.shape[0]} строк на {final_df.shape[1]} колонок.")

Экспорт завершен! Готовая таблица лежит в: ../data/output/guns_result_20260719_011753.csv
Размерность: 133 строк на 14 колонок.
